In [93]:
import pandas as pd
import numpy as np
import re

In [94]:
df = pd.read_csv(r"C:\Users\A-FF-L1-D07\Desktop\05_Drugs_الأدوية-20260905T153828Z-1-001\05_Drugs_الأدوية\01_plant_to_drug_to_disease.csv")

In [95]:
print(df.shape)
print(df.info())
print(df.isna().sum())        # لمعرفة القيم الفارغة في كل عمود
df.head()

(114, 11)
<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   kegg_drug       114 non-null    str    
 1   drug_name       114 non-null    str    
 2   match_type      114 non-null    str    
 3   plant_ids       114 non-null    str    
 4   pubchem_cids    0 non-null      float64
 5   plant_names     114 non-null    str    
 6   plant_names_ar  114 non-null    str    
 7   atc_codes       63 non-null     str    
 8   diseases        0 non-null      float64
 9   atc_class       63 non-null     str    
 10  confidence      114 non-null    str    
dtypes: float64(2), str(9)
memory usage: 9.9 KB
None
kegg_drug           0
drug_name           0
match_type          0
plant_ids           0
pubchem_cids      114
plant_names         0
plant_names_ar      0
atc_codes          51
diseases          114
atc_class          51
confidence          0
dtype: int64


,kegg_drug,drug_name,match_type,plant_ids,pubchem_cids,plant_names,plant_names_ar,atc_codes,diseases,atc_class,confidence
0,D00064,l-Menthol (JP19); Levomenthol,partial,P13,NaN,Mentha piperita,نعناع فلفلي,NaN,NaN,NaN,medium
1,D00113,Atropine (USP); Atropen (TN),exact,P06;P08,NaN,Atropa belladonna;Datura stramonium,ست الحسن;داتورة,A03BA01;S01FA01,NaN,الجهاز الهضمي والأيض;الحواس,high
2,D00138,Scopolamine (INN); Transderm scop (TN),exact,P07;P08,NaN,Hyoscyamus muticus;Datura stramonium,سكران مصري;داتورة,A04AD01;N05CM05;S01FA02,NaN,الجهاز العصبي;الجهاز الهضمي والأيض;الحواس,high
3,D00139,Methoxsalen (JP19/USP); Oxsoralen (TN); UVADEX...,exact,P05,NaN,Ammi majus,خلة شيطاني,D05AD02;D05BA02,NaN,الجلد,high
4,D00147,Hyoscyamine (USP),exact,P06,NaN,Atropa belladonna,ست الحسن,A03BA03,NaN,الجهاز الهضمي والأيض,high


In [96]:
print(df['kegg_drug'].isna().sum())
print(df['kegg_drug'].duplicated().sum())
df['kegg_drug'] = df['kegg_drug'].str.strip()

0
0


In [97]:
def extract_drug_info(text):
    if pd.isna(text):
        return pd.Series([None, None])
    
    parts = [p.strip() for p in text.split(';')]
    
    # 1. Primary scientific name (cleaned from brackets like USP, INN)
    primary_name = re.sub(r'\s*\([^)]*\)', '', parts[0]).strip()
    
    # 2. Extract remaining synonyms and trade names (if any)
    if len(parts) > 1:
        synonyms = [re.sub(r'\s*\([^)]*\)', '', p).strip() for p in parts[1:]]
        synonyms_str = "; ".join(synonyms)
    else:
        synonyms_str = None
        
    return pd.Series([primary_name, synonyms_str])

# Create new extracted columns
df[['primary_drug_name', 'synonyms']] = df['drug_name'].apply(extract_drug_info)

In [98]:
df['match_type'] = df['match_type'].astype('category')

print(df['match_type'].value_counts())

match_type
partial    80
exact      29
curated     5
Name: count, dtype: int64


In [99]:
df['plant_ids_list'] = df['plant_ids'].str.split(';').apply(lambda x: [i.strip() for i in x])

# Option B: Reshape dataframe into Tidy format for complete statistical modeling
df_tidy = df.assign(plant_id_single=df['plant_ids'].str.split(';')).explode('plant_id_single')
df_tidy['plant_id_single'] = df_tidy['plant_id_single'].str.strip()

# Verification
print("Original Rows:", len(df))
print("Tidy Rows (Expanded):", len(df_tidy))
print("\nTop 5 Most Frequent Plant IDs:")
print(df_tidy['plant_id_single'].value_counts().head())

Original Rows: 114
Tidy Rows (Expanded): 141

Top 5 Most Frequent Plant IDs:
plant_id_single
P10    40
P08    26
P06    23
P13    15
P07     7
Name: count, dtype: int64


In [100]:
# Option 1: Drop the completely empty column (Recommended for general analysis)
df_clean = df.drop(columns=['pubchem_cids'])

# Verify drop
print("Remaining columns count:", len(df_clean.columns))

Remaining columns count: 13


In [101]:
# Convert semi-colon separated text into a clean Python list
df['plant_names_list'] = df['plant_names'].str.split(';').apply(lambda x: [i.strip() for i in x])

# Display sample output
print(df[['kegg_drug', 'plant_ids', 'plant_names', 'plant_names_list']].head(4))

  kegg_drug plant_ids                           plant_names  \
0    D00064       P13                       Mentha piperita   
1    D00113   P06;P08   Atropa belladonna;Datura stramonium   
2    D00138   P07;P08  Hyoscyamus muticus;Datura stramonium   
3    D00139       P05                            Ammi majus   

                          plant_names_list  
0                        [Mentha piperita]  
1   [Atropa belladonna, Datura stramonium]  
2  [Hyoscyamus muticus, Datura stramonium]  
3                             [Ammi majus]  


In [102]:
# Convert semi-colon separated Arabic names into a clean Python list
df['plant_names_ar_list'] = df['plant_names_ar'].str.split(';').apply(lambda x: [i.strip() for i in x])

# Display sample result
print(df[['kegg_drug', 'plant_names', 'plant_names_ar', 'plant_names_ar_list']].head(4))

  kegg_drug                           plant_names     plant_names_ar  \
0    D00064                       Mentha piperita        نعناع فلفلي   
1    D00113   Atropa belladonna;Datura stramonium    ست الحسن;داتورة   
2    D00138  Hyoscyamus muticus;Datura stramonium  سكران مصري;داتورة   
3    D00139                            Ammi majus         خلة شيطاني   

    plant_names_ar_list  
0         [نعناع فلفلي]  
1    [ست الحسن, داتورة]  
2  [سكران مصري, داتورة]  
3          [خلة شيطاني]  


In [103]:
# 1. Fill missing values with 'Unclassified'
df['atc_codes_clean'] = df['atc_codes'].fillna('Unclassified')

# 2. Convert semi-colon separated string into a clean list
df['atc_codes_list'] = df['atc_codes_clean'].str.split(';').apply(lambda x: [i.strip() for i in x])

# Display sample output
print(df[['kegg_drug', 'atc_codes', 'atc_codes_clean', 'atc_codes_list']].head(6))

  kegg_drug                atc_codes          atc_codes_clean  \
0    D00064                      NaN             Unclassified   
1    D00113          A03BA01;S01FA01          A03BA01;S01FA01   
2    D00138  A04AD01;N05CM05;S01FA02  A04AD01;N05CM05;S01FA02   
3    D00139          D05AD02;D05BA02          D05AD02;D05BA02   
4    D00147                  A03BA03                  A03BA03   
5    D00157                  A05BA08                  A05BA08   

                atc_codes_list  
0               [Unclassified]  
1           [A03BA01, S01FA01]  
2  [A04AD01, N05CM05, S01FA02]  
3           [D05AD02, D05BA02]  
4                    [A03BA03]  
5                    [A05BA08]  


In [104]:
# Drop completely empty columns
df_clean = df.drop(columns=['diseases'])

In [105]:
# 1. Fill missing values
df['atc_class_clean'] = df['atc_class'].fillna('غير محدد')

# 2. Split into a clean Python list
df['atc_class_list'] = df['atc_class_clean'].str.split(';').apply(lambda x: [i.strip() for i in x])

# Display sample output
print(df[['kegg_drug', 'atc_codes', 'atc_class', 'atc_class_list']].head(4))

  kegg_drug                atc_codes  \
0    D00064                      NaN   
1    D00113          A03BA01;S01FA01   
2    D00138  A04AD01;N05CM05;S01FA02   
3    D00139          D05AD02;D05BA02   

                                   atc_class  \
0                                        NaN   
1                الجهاز الهضمي والأيض;الحواس   
2  الجهاز العصبي;الجهاز الهضمي والأيض;الحواس   
3                                      الجلد   

                                  atc_class_list  
0                                     [غير محدد]  
1                 [الجهاز الهضمي والأيض, الحواس]  
2  [الجهاز العصبي, الجهاز الهضمي والأيض, الحواس]  
3                                        [الجلد]  


In [106]:
#Define ordered categorical type for confidence
confidence_levels = ['low', 'medium', 'high']
df['confidence'] = pd.Categorical(df['confidence'], categories=confidence_levels, ordered=True)

In [107]:
def fix_arabic_encoding(text):
    if pd.isna(text):
        return text
    try:
        # إعادة تحويل النص إلى بايتات بترميز latin1/cp1252 ثم فكه بترميز utf-8
        return str(text).encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        try:
            return str(text).encode('cp1252').decode('utf-8')
        except Exception:
            return text

# تطبيق الدالة على العمود
df['plant_names_ar'] = df['plant_names_ar'].apply(fix_arabic_encoding)

# معاينة النتيجة
print(df['plant_names_ar'].head())

0          نعناع فلفلي
1      ست الحسن;داتورة
2    سكران مصري;داتورة
3           خلة شيطاني
4             ست الحسن
Name: plant_names_ar, dtype: str


In [108]:
def fix_encoding(text):
    if pd.isna(text):
        return text
    try:
        return str(text).encode('cp1252').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        try:
            return str(text).encode('latin1').decode('utf-8')
        except Exception:
            return text

# تطبيق الإصلاح على عمود atc_class
df['atc_class'] = df['atc_class'].apply(fix_encoding)

# معاينة النتيجة بعد الإصلاح
print(df['atc_class'].head())

0                                          NaN
1                  الجهاز الهضمي والأيض;الحواس
2    الجهاز العصبي;الجهاز الهضمي والأيض;الحواس
3                                        الجلد
4                         الجهاز الهضمي والأيض
Name: atc_class, dtype: str


In [109]:
cols_to_split = ['plant_ids', 'plant_names', 'plant_names_ar']

for col in cols_to_split:
    df[col] = df[col].apply(
        lambda x: [i.strip() for i in str(x).split(';')] if pd.notna(x) else []
    )

# 4. توسيع الصفوف: كل نبات في سطر لوحده (drug_name بيتكرر كامل زي ما هو)
df_tidy = df.explode(cols_to_split).reset_index(drop=True)

# 5. الحفظ فوق نفس اسم الملف الأصلي (مش ملف جديد)
df_tidy.to_csv('cleaned_plant_to_drug1.csv', index=False, encoding='utf-8-sig')

print('تم الحفظ فوق الملف الأصلي بنجاح ✅')
print('عدد الصفوف بعد الفصل:', len(df_tidy))

تم الحفظ فوق الملف الأصلي بنجاح ✅
عدد الصفوف بعد الفصل: 141


In [110]:
# Save cleaned version
df_clean.to_csv('cleaned_plant_to_drug1.csv', index=False, encoding='utf-8-sig')